# sys1 v1: LoRA fine-tune

Run this notebook on a free Colab/Kaggle GPU (Runtime > Change runtime type > GPU).

Steps:
1. Upload `data/prepared/train.jsonl` and `data/prepared/val.jsonl` (produced
   locally by `training/prepare_dataset.py`) to this Colab session.
2. Run all cells. This LoRA fine-tunes the base model to answer judgment
   prompts with the correct letter more reliably.
3. Download the resulting adapter folder (`sys1-lora-out/`) back to your
   machine and point `src/sys1/engine.py`'s `Engine` at it (see bottom cell).

In [ ]:
!pip install -q -U transformers accelerate peft trl datasets bitsandbytes

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"  # bump to 1.5B/3B once this works
TRAIN_FILE = "train.jsonl"
VAL_FILE = "val.jsonl"
OUT_DIR = "sys1-lora-out"

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "validation": VAL_FILE},
)
dataset

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto")

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=OUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    bf16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
)
trainer.train()

In [ ]:
trainer.save_model(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
# Zip and download OUT_DIR, then point Engine(model_name=OUT_DIR) at the
# extracted folder locally, OR merge the adapter into the base model with
# model.merge_and_unload() first if you want a single standalone checkpoint.